# The 14th Homework

## Импорты, seed и среда

In [1]:
import os
import sys
import random
import subprocess
from typing import List, Dict, Tuple, Optional
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

import numpy as np
import pandas as pd
import torch

import re
from pathlib import Path
from collections import Counter

import frontmatter

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

os.environ["TOKENIZERS_PARALLELISM"] = "false"

FAISS_AVAILABLE = True


print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS available:", FAISS_AVAILABLE)

/Users/polina/vscode/art_ing_course/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NumPy: 2.4.4
Pandas: 3.0.2
FAISS available: True


In [2]:
SEED = 42
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)


DEVICE = "mps" if torch.mps.is_available() else "cpu"

print("Устройство для работы:", DEVICE)

Устройство для работы: mps


## База знаний и первичный анализ

In [3]:
def load_kb(kb_dir: str) -> pd.DataFrame:
    records = []
    path = Path(kb_dir)

    for md_file in sorted(path.rglob("*.md")):
        post = frontmatter.load(md_file)       # парсим YAML + тело
        meta = post.metadata
        body = post.content.strip()

        records.append({
            # путь
            "filename":    md_file.name,
            "rel_path":    str(md_file.relative_to(path)),
            # метаданные из YAML
            "title":       meta.get("title", ""),
            "category":    meta.get("category", ""),
            "subcategory": meta.get("subcategory", ""),
            "tags":        meta.get("tags", []),
            "skin_type":   meta.get("skin_type", []),
            "skin_concern":meta.get("skin_concern", []),
            # статистика
            "body":        body,
            "n_words":     len(body.split()),
            "n_chars":     len(body),
            "n_h2":        len(re.findall(r"^##\s+", body, re.M)),
        })

    return pd.DataFrame(records)

In [4]:
df = load_kb("./knowledge_base/skincare_kb")

print(f"Загружено документов: {len(df)}")
df.sample(10)

Загружено документов: 31


,filename,rel_path,title,category,subcategory,tags,skin_type,skin_concern,body,n_words,n_chars,n_h2
27,04_split_ends.md,04_hair_care/03_hair_problems/04_split_ends.md,Секущиеся концы — причины и лечение,уход_за_волосами,проблемы_волос,"[секущиеся_концы, термозащита, кератин, повреж...",[],[],# Секущиеся концы: причины и методы борьбы\n\n...,200,1593,2
15,01_dry_skin.md,03_skincare_by_type_and_concern/01_skin_types/...,Сухая кожа,уход_за_кожей,типы_кожи,"[сухая_кожа, гидратация, барьер, шелушение, ув...",[сухая],"[сухость, обезвоживание, тусклость]",# Dry Skin\n\n## Что такое сухая кожа\n\nСухая...,363,2412,5
23,01_hair_and_scalp_types.md,04_hair_care/01_hair_and_scalp_types.md,Типы волос и кожи головы,уход_за_волосами,типы,"[волосы, типы_волос, кожа_головы, кудрявые, пр...",[],[],# Типы волос: Ирландские локоны\n\n**Теги:** #...,119,859,2
17,03_combination_skin.md,03_skincare_by_type_and_concern/01_skin_types/...,Комбинированная кожа,уход_за_кожей,типы_кожи,"[комбинированная_кожа, т-зона, поры, баланс, у...",[комбинированная],"[акне, поры, сухость, обезвоживание]",# Combination Skin\n\n## Что такое комбинирова...,329,2095,5
8,04_spf_protection.md,02_cosmetics_and_ingredients/01_cosmetic_produ...,Защита от солнца (SPF),косметика,типы_средств,"[spf, солнцезащита, uva, uvb, фотостарение, пи...",[все типы],"[пигментация, морщины, фотостарение]",# Защита от солнца (SPF)\n\n## Правила использ...,264,1947,1
9,01_retinol_and_derivatives.md,02_cosmetics_and_ingredients/02_active_ingredi...,Ретинол и его производные,косметика,активные_ингредиенты,"[ретинол, ретиноиды, антиэйдж, акне, витамин_А]","[жирная, комбинированная, зрелая]","[акне, морщины, пигментация, поры]",# Ретинол и его производные (Ретиноиды)\n\n## ...,256,2017,4
29,01_sos_skin_detox.md,06_procedures_and_techniques/01_sos_skin_detox.md,SOS-детокс кожи после праздников,процедуры,ситуативный_уход,"[детокс, восстановление, отеки, воспаления, об...",[все типы],"[обезвоживание, тусклость, отеки, воспаления]",# SOS-Детокс для кожи (после вечеринок и празд...,151,1062,3
24,02_curly_girl_method.md,04_hair_care/02_curly_girl_method.md,Ирландские локоны — тип текстуры волос,уход_за_волосами,типы_волос,"[ирландские_локоны, кудри, текстура_волос, куд...",[],[],# Ирландские локоны\n\n## Что такое «Ирландски...,133,929,3
12,04_copper_peptides.md,02_cosmetics_and_ingredients/02_active_ingredi...,Пептиды меди в косметике,косметика,активные_ингредиенты,"[пептиды, пептиды_меди, антиэйдж, коллаген, ан...","[зрелая, нормальная, комбинированная]","[морщины, потеря_упругости, воспаления]",# Пептиды меди в косметике (Copper Peptides)\n...,172,1251,4
0,01_healthy_eating_basics.md,01_nutrition_and_diets/01_healthy_eating_basic...,Основы здорового питания для красоты и здоровья,питание,основы,"[питание, рацион, здоровье, антиоксиданты, вит...",[все типы],"[тусклость, сухость, акне]",# Основы здорового питания для красоты и здоро...,196,1486,1


In [5]:
df[df["n_chars"] < 500][["filename", "n_chars"]]

,filename,n_chars


In [6]:
df[df["title"] == ""][["filename", "rel_path"]]

,filename,rel_path


In [7]:
df[df["title"].duplicated(keep=False)][["filename", "title"]]

,filename,title


In [8]:
df[df["n_h2"] == 0][["filename", "category"]]

,filename,category


In [9]:
for i in range (5):
    print(f"title: {df['title'].iloc[i]} \ntext - {df['body'][:10]}")

title: Основы здорового питания для красоты и здоровья 
text - 0    # Основы здорового питания для красоты и здоро...
1    # Продукты, полезные для кожи\n#полезные_проду...
2    # Продукты, вредные для кожи\n#вредные_продукт...
3    # Питьевой режим и гидратация\n#питьевой_режим...
4    # Диета при акне (Anti-acne протокол)\n#диета_...
5    # Очищение кожи: демакияж и умывание\n#очищени...
6    # Тонизирование: Тоник, Тонер и Эссенция\n\n##...
7    # Увлажнение кожи: кремы и сыворотки\n#увлажня...
8    # Защита от солнца (SPF)\n\n## Правила использ...
9    # Ретинол и его производные (Ретиноиды)\n\n## ...
Name: body, dtype: str
title: Продукты, полезные для кожи 
text - 0    # Основы здорового питания для красоты и здоро...
1    # Продукты, полезные для кожи\n#полезные_проду...
2    # Продукты, вредные для кожи\n#вредные_продукт...
3    # Питьевой режим и гидратация\n#питьевой_режим...
4    # Диета при акне (Anti-acne протокол)\n#диета_...
5    # Очищение кожи: демакияж и умывание\n#оч

База знаний корректна. Выбрана предметная область - уход за внешностью. База знаний полностью покрывает возможные вопросы по этой теме, при этом вписывается в размер. Также в базе знаний настроен фильтр по категориям, заголовки и тп

## Чанкинг

In [10]:
HEADERS_TO_SPLIT = [
    ("#",   "h1"),   # заголовок документа
    ("##",  "h2"),   # основные секции → главные чанки
    ("###", "h3"),   # подсекции → дочерние чанки
]
MAX_CHUNK_SIZE = 1000  # символов — страховка от гигантских секций
CHUNK_OVERLAP  = 100   # 10% от 1000 — контекст между чанками

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=HEADERS_TO_SPLIT,
    strip_headers=False   # заголовок остаётся в тексте чанка
)

# Страховочный сплиттер для слишком длинных секций
char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=MAX_CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

In [11]:
all_chunks = []

for _, row in df.iterrows():
    # 1. Режем по заголовкам ##
    header_chunks = md_splitter.split_text(row["body"])

    for chunk in header_chunks:
        # 2. Если секция длиннее MAX_CHUNK_SIZE — дробим дополнительно
        if len(chunk.page_content) > MAX_CHUNK_SIZE:
            sub_chunks = char_splitter.split_documents([chunk])
        else:
            sub_chunks = [chunk]

        for sc in sub_chunks:
            # 3. Добавляем метаданные из YAML-шапки
            sc.metadata.update({
                "source":      row["filename"],
                "title":       row["title"],
                "category":    row["category"],
                "subcategory": row["subcategory"],
                "skin_type":   row["skin_type"],
                "skin_concern":row["skin_concern"],
                "tags":        row["tags"],
            })
            all_chunks.append(sc)

print(f"Документов:  {len(df)}")
print(f"Чанков всего: {len(all_chunks)}")
print(f"Среднее на документ: {len(all_chunks)/len(df):.1f}")

Документов:  31
Чанков всего: 166
Среднее на документ: 5.4


In [12]:
# Выбираем один документ для наглядности
doc_name = '01_skin_beneficial_foods.md'
doc_chunks = [c for c in all_chunks if c.metadata["source"] == doc_name]

print(f"📄 {doc_name} → {len(doc_chunks)} чанков\n")
for i, c in enumerate(doc_chunks):
    section = c.metadata.get("h2", c.metadata.get("h1", "—"))
    print(f"  Чанк {i+1}: [{section}]  {len(c.page_content)} симв.")
    print(f"  {c.page_content[:120].strip()}...")
    print()

📄 01_skin_beneficial_foods.md → 3 чанков

  Чанк 1: [Влияние продуктов на состояние кожи]  324 симв.
  # Продукты, полезные для кожи
#полезные_продукты, #продукты_для_кожи, #омега3, #антиоксиданты, #коллаген_из_пищи  
## Вл...

  Чанк 2: [Влияние продуктов на состояние кожи]  898 симв.
  ### Топ-компоненты и их пищевые источники
- **Омега-3 жирные кислоты:** Снимают воспаление, укрепляют клеточные мембраны...

  Чанк 3: [Влияние продуктов на состояние кожи]  286 симв.
  ### Рекомендации для внедрения
- Добавляйте порцию жирной рыбы 2-3 раза в неделю.
- Ежедневно употребляйте минимум 400 г...



## Эмбеддинги и индекс FAISS

In [13]:
embedding_model = HuggingFaceEmbeddings(
    model_name="deepvk/USER-base",
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True},  # обязательно для cosine similarity
)

# Быстрая проверка
test_vec = embedding_model.embed_query("тест")
print(f"Размерность вектора: {len(test_vec)}")  # → 768

Loading weights: 100%|██████████| 160/160 [00:00<00:00, 10155.55it/s]
Default prompt name is set to 'query'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.


Размерность вектора: 768


In [14]:
# all_chunks уже есть из шага чанкинга
# Убедимся что метаданные корректны — списки → строки (FAISS не хранит списки)

def normalize_metadata(meta: dict) -> dict:
    """FAISS/LangChain требует строки в metadata, не списки"""
    result = {}
    for k, v in meta.items():
        if isinstance(v, list):
            result[k] = ", ".join(str(i) for i in v)
        else:
            result[k] = str(v) if v is not None else ""
    return result

# Нормализуем metadata в каждом чанке
for chunk in all_chunks:
    chunk.metadata = normalize_metadata(chunk.metadata)

print(f"Чанков готово к индексации: {len(all_chunks)}")
print(f"\nПример метаданных чанка:")
for k, v in all_chunks[0].metadata.items():
    print(f"  {k}: {v}")

Чанков готово к индексации: 166

Пример метаданных чанка:
  h1: Основы здорового питания для красоты и здоровья
  h2: Базовые принципы нутрициологии
  source: 01_healthy_eating_basics.md
  title: Основы здорового питания для красоты и здоровья
  category: питание
  subcategory: основы
  skin_type: все типы
  skin_concern: тусклость, сухость, акне
  tags: питание, рацион, здоровье, антиоксиданты, витамины


In [15]:
from langchain_community.vectorstores import FAISS

print("Строим индекс... (займёт ~30–60 сек)")

vectorstore = FAISS.from_documents(
    documents=all_chunks,
    embedding=embedding_model,
)

# Сохраняем на диск — чтобы не пересчитывать каждый раз
vectorstore.save_local("./faiss_index")
print(f"✅ Индекс сохранён: ./faiss_index/")
print(f"   Векторов в индексе: {vectorstore.index.ntotal}")

Строим индекс... (займёт ~30–60 сек)
✅ Индекс сохранён: ./faiss_index/
   Векторов в индексе: 166


In [16]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

queries = [
    "как ухаживать за сухой кожей зимой",
    "какие кислоты помогают от акне",
    "что такое постакне и как от него избавиться",
    "какие продукты улучшают состояние кожи",
    "как правильно наносить SPF",
]

for query in queries:
    print(f"\n{'─'*60}")
    print(f"🔍 Запрос: {query}")
    print(f"{'─'*60}")
    results = retriever.invoke(query)

    for i, doc in enumerate(results, 1):
        meta = doc.metadata
        section = meta.get("h2", meta.get("h1", "—"))
        print(f"\n  [{i}] {meta.get('title')}  →  {section}")
        print(f"      категория: {meta.get('category')} / {meta.get('subcategory')}")
        print(f"      {doc.page_content[:200].strip()}...")


────────────────────────────────────────────────────────────
🔍 Запрос: как ухаживать за сухой кожей зимой
────────────────────────────────────────────────────────────

  [1] Подготовка кожи к весне  →  Подготовка кожи к весне
      категория: процедуры / сезонный_уход
      # Подготовка кожи к весне  
Переход от плотных зимних текстур к более легким и защитным форматам из-за перепадов температур и активного солнца....

  [2] Реакция кожи на холод — зимний уход  →  Реакция кожи на холод: «Светофор» состояний и уход
      категория: уход_за_кожей / проблемы_кожи
      # Реакция кожи на холод: «Светофор» состояний и уход  
Перепады температур, мороз, ветер и сухой воздух в помещениях разрушают гидролипидную мантию кожи. Диагностика по принципу «светофора»:...

  [3] Сезонные и SOS-ритуалы ухода  →  Подготовка кожи к весне
      категория: процедуры / ритуалы
      ## Подготовка кожи к весне  
Переход от плотных зимних текстур к более легким и защитным форматам из-за перепадов температур 

In [17]:
# similarity_search_with_score возвращает L2-дистанцию
# Чем меньше — тем ближе (при normalize_embeddings=True)

query = "ретинол против морщин: с чего начать"
results_with_scores = vectorstore.similarity_search_with_score(query, k=5)

print(f"Запрос: {query}\n")
for doc, score in results_with_scores:
    print(f"  score={score:.4f}  |  {doc.metadata.get('title')}  →  {doc.metadata.get('h2','—')}")

Запрос: ретинол против морщин: с чего начать

  score=0.7316  |  Ретинол и его производные  →  Что такое ретинол и как он работает
  score=0.8479  |  Морщины и потеря упругости  →  Ошибки в антивозрастном уходе
  score=0.8946  |  Морщины и потеря упругости  →  Ключевые активы
  score=0.9026  |  Ретинол и его производные  →  Правила ввода ретинола в уход (Лестница ретинола)
  score=0.9143  |  Комбинированная кожа  →  Ингредиенты


## Контрольные запросы и оценка retrieval

In [18]:
EVAL_QUERIES = [
    {
        "id": "Q01",
        "query": "как ухаживать за сухой кожей зимой",
        "relevant_docs": ["01_dry_skin.md"],
        "relevant_keywords": ["сухая", "барьер", "церамиды", "шелушение"],
        "note": "Прямое совпадение с документом типа кожи"
    },
    {
        "id": "Q02",
        "query": "какие кислоты помогают от акне и расширенных пор",
        "relevant_docs": ["03_aha_bha_acids.md", "01_acne_and_post_acne.md"],
        "relevant_keywords": ["bha", "салициловая", "акне", "поры"],
        "note": "Два релевантных документа"
    },
    {
        "id": "Q03",
        "query": "что такое постакне и как убрать тёмные пятна",
        "relevant_docs": ["01_acne_and_post_acne.md", "02_pigmentation.md"],
        "relevant_keywords": ["постакне", "пигментация", "пятна", "PIH"],
        "note": "Два тематически близких документа"
    },
    {
        "id": "Q04",
        "query": "продукты и питание для здоровой кожи",
        "relevant_docs": ["01_skin_beneficial_foods.md", "01_healthy_eating_basics.md"],
        "relevant_keywords": ["продукты", "питание", "омега", "антиоксидант"],
        "note": "Раздел питания"
    },
    {
        "id": "Q05",
        "query": "как правильно наносить и выбирать SPF крем",
        "relevant_docs": ["04_spf_protection.md"],
        "relevant_keywords": ["spf", "солнцезащита", "uva", "uvb"],
        "note": "Узкий конкретный запрос"
    },
    {
        "id": "Q06",
        "query": "ретинол с чего начать и как избежать раздражения",
        "relevant_docs": ["01_retinol_and_derivatives.md"],
        "relevant_keywords": ["ретинол", "ретинизация", "раздражение", "концентрация"],
        "note": "Запрос про конкретный ингредиент"
    },
    {
        "id": "Q07",
        "query": "уход за жирной кожей матирование и поры",
        "relevant_docs": ["02_oily_skin.md"],
        "relevant_keywords": ["жирная", "себум", "матирование", "ниацинамид"],
        "note": "Прямое совпадение тип кожи"
    },
    {
        "id": "Q08",
        "query": "витамин С сыворотка осветление пигментных пятен",
        "relevant_docs": ["02_vitamin_c.md", "02_pigmentation.md"],
        "relevant_keywords": ["витамин с", "аскорбиновая", "осветление", "тирозиназа"],
        "note": "Смешанный: ингредиент + проблема"
    },
    {
        "id": "Q09",
        "query": "выпадение волос причины и что делать",
        "relevant_docs": ["01_hair_loss.md"],
        "relevant_keywords": ["выпадение", "алопеция", "волосы", "трихология"],
        "note": "Раздел волос — отдельная категория"
    },
    {
        "id": "Q10",
        "query": "как читать состав косметики и что такое INCI",
        "relevant_docs": ["03_reading_inci_labels.md"],
        "relevant_keywords": ["inci", "состав", "этикетка", "ингредиенты"],
        "note": "Точный узкий запрос"
    },
]

print(f"Контрольных запросов: {len(EVAL_QUERIES)}")
for q in EVAL_QUERIES:
    print(f"  {q['id']}  {q['query'][:55]:<55}  → {q['relevant_docs']}")


Контрольных запросов: 10
  Q01  как ухаживать за сухой кожей зимой                       → ['01_dry_skin.md']
  Q02  какие кислоты помогают от акне и расширенных пор         → ['03_aha_bha_acids.md', '01_acne_and_post_acne.md']
  Q03  что такое постакне и как убрать тёмные пятна             → ['01_acne_and_post_acne.md', '02_pigmentation.md']
  Q04  продукты и питание для здоровой кожи                     → ['01_skin_beneficial_foods.md', '01_healthy_eating_basics.md']
  Q05  как правильно наносить и выбирать SPF крем               → ['04_spf_protection.md']
  Q06  ретинол с чего начать и как избежать раздражения         → ['01_retinol_and_derivatives.md']
  Q07  уход за жирной кожей матирование и поры                  → ['02_oily_skin.md']
  Q08  витамин С сыворотка осветление пигментных пятен          → ['02_vitamin_c.md', '02_pigmentation.md']
  Q09  выпадение волос причины и что делать                     → ['01_hair_loss.md']
  Q10  как читать состав косметики и что такое INCI    

In [19]:
def hit_at_k(retrieved_docs, relevant_filenames, relevant_keywords, k):
    """
    Hit@k = 1 если хотя бы один из top-k чанков
    принадлежит релевантному документу ИЛИ содержит ключевые слова.
    """
    for doc in retrieved_docs[:k]:
        source = doc.metadata.get("source", "")
        content = doc.page_content.lower()
        # Проверка по имени файла
        if any(rel in source for rel in relevant_filenames):
            return 1
        # Запасная проверка по ключевым словам
        if any(kw.lower() in content for kw in relevant_keywords):
            return 1
    return 0


def recall_at_k(retrieved_docs, relevant_filenames, relevant_keywords, k):
    """
    Recall@k = доля найденных релевантных документов из всех ожидаемых.
    Считаем по именам файлов.
    """
    found = set()
    for doc in retrieved_docs[:k]:
        source = doc.metadata.get("source", "")
        content = doc.page_content.lower()
        for rel in relevant_filenames:
            if rel in source:
                found.add(rel)
        # Запасной критерий: ключевые слова
        if any(kw.lower() in content for kw in relevant_keywords):
            for rel in relevant_filenames:
                found.add(rel)  # засчитываем как покрытие
    return len(found) / len(relevant_filenames) if relevant_filenames else 0


def mrr_at_k(retrieved_docs, relevant_filenames, relevant_keywords, k):
    """
    MRR@k = 1/rank первого релевантного результата.
    """
    for rank, doc in enumerate(retrieved_docs[:k], start=1):
        source = doc.metadata.get("source", "")
        content = doc.page_content.lower()
        if any(rel in source for rel in relevant_filenames):
            return 1 / rank
        if any(kw.lower() in content for kw in relevant_keywords):
            return 1 / rank
    return 0.0


print("Функции метрик определены: hit_at_k, recall_at_k, mrr_at_k")

Функции метрик определены: hit_at_k, recall_at_k, mrr_at_k


In [20]:
K = 5  # top-k для оценки

results = []

for q in EVAL_QUERIES:
    retrieved = vectorstore.similarity_search(q["query"], k=K)

    hit   = hit_at_k(retrieved, q["relevant_docs"], q["relevant_keywords"], k=K)
    rec   = recall_at_k(retrieved, q["relevant_docs"], q["relevant_keywords"], k=K)
    mrr   = mrr_at_k(retrieved, q["relevant_docs"], q["relevant_keywords"], k=K)

    # Детали топ-1 результата
    top1 = retrieved[0] if retrieved else None
    top1_source  = top1.metadata.get("source", "—")  if top1 else "—"
    top1_section = top1.metadata.get("h2",     "—")  if top1 else "—"

    results.append({
        "id":           q["id"],
        "query":        q["query"],
        "hit@k":        hit,
        "recall@k":     round(rec, 2),
        "mrr@k":        round(mrr, 2),
        "top1_source":  top1_source,
        "top1_section": top1_section,
        "expected":     str(q["relevant_docs"]),
        "note":         q["note"],
    })

df_eval = pd.DataFrame(results)

# ── Итоговые метрики ──────────────────────────────────────────
mean_hit    = df_eval["hit@k"].mean()
mean_recall = df_eval["recall@k"].mean()
mean_mrr    = df_eval["mrr@k"].mean()

print(f"\n{'═'*60}")
print(f"  K = {K}")
print(f"  Hit@{K}    = {mean_hit:.3f}   ({int(mean_hit * len(df_eval))}/{len(df_eval)} запросов)")
print(f"  Recall@{K} = {mean_recall:.3f}")
print(f"  MRR@{K}   = {mean_mrr:.3f}")
print(f"{'═'*60}")


════════════════════════════════════════════════════════════
  K = 5
  Hit@5    = 1.000   (10/10 запросов)
  Recall@5 = 1.000
  MRR@5   = 0.925
════════════════════════════════════════════════════════════


In [22]:
display_cols = ["id", "query", "hit@k", "recall@k", "mrr@k", "top1_source"]

def color_hit(v):
    if v == 1:
        return "background-color: #c8e6c9"  # зелёный
    elif v == 0:
        return "background-color: #ffcdd2"  # красный
    return ""

display(
    df_eval[display_cols].style
    .map(color_hit, subset=["hit@k"])
    .format({"recall@k": "{:.2f}", "mrr@k": "{:.2f}"})
)

,id,query,hit@k,recall@k,mrr@k,top1_source
0,Q01,как ухаживать за сухой кожей зимой,1,1.00,0.25,02_spring_skin_prep.md
1,Q02,какие кислоты помогают от акне и расширенных пор,1,1.00,1.00,02_acne_diet.md
2,Q03,что такое постакне и как убрать тёмные пятна,1,1.00,1.00,01_acne_and_post_acne.md
3,Q04,продукты и питание для здоровой кожи,1,1.00,1.00,01_healthy_eating_basics.md
4,Q05,как правильно наносить и выбирать SPF крем,1,1.00,1.00,04_spf_protection.md
5,Q06,ретинол с чего начать и как избежать раздражения,1,1.00,1.00,01_retinol_and_derivatives.md
6,Q07,уход за жирной кожей матирование и поры,1,1.00,1.00,02_oily_skin.md
7,Q08,витамин С сыворотка осветление пигментных пятен,1,1.00,1.00,02_pigmentation.md
8,Q09,выпадение волос причины и что делать,1,1.00,1.00,04_split_ends.md
9,Q10,как читать состав косметики и что такое INCI,1,1.00,1.00,03_reading_inci_labels.md


In [23]:
failures = df_eval[df_eval["hit@k"] == 0]

if failures.empty:
    print("✅ Все запросы получили хотя бы один релевантный результат в top-k")
else:
    print(f"❌ Провальных запросов: {len(failures)}\n")
    for _, row in failures.iterrows():
        print(f"  {row['id']}: {row['query']}")
        print(f"    Ожидалось:  {row['expected']}")
        print(f"    Получено:   {row['top1_source']}  |  {row['top1_section']}")
        print()

        # Показываем top-5 для этого запроса подробно
        retrieved = vectorstore.similarity_search(row["query"], k=K)
        print(f"    Top-{K} результатов:")
        for i, doc in enumerate(retrieved, 1):
            s = doc.metadata.get("source", "—")
            h = doc.metadata.get("h2", doc.metadata.get("h1", "—"))
            score_info = f"{len(doc.page_content)} симв."
            print(f"      {i}. {s:<40} [{h[:40]}]  {score_info}")
        print()



✅ Все запросы получили хотя бы один релевантный результат в top-k


In [24]:
df_eval.to_csv("retrieval_eval_results.csv", index=False, encoding="utf-8")
print("💾 Сохранено: retrieval_eval_results.csv")
print(f"\nИтог: Hit@{K}={mean_hit:.2f}  Recall@{K}={mean_recall:.2f}  MRR@{K}={mean_mrr:.2f}")

💾 Сохранено: retrieval_eval_results.csv

Итог: Hit@5=1.00  Recall@5=1.00  MRR@5=0.93
